# <font color=#0099CC>**Portfolio — Inference (run on Colab)**</font>

This notebook runs on **Colab** (TensorFlow + GPU). It:

1. Loads a trained model checkpoint (CNN, MLP, RNN, or Mixto).
2. Reconstructs the exact scaler from the training partition.
3. At each rebalancing date, predicts and computes expected returns.
4. Saves `expected_returns.csv` for the local backtest notebook.

**To switch models**: change `ACTIVE_MODEL` in cell 2.

---
## <font color=#0099CC>**1. ENVIRONMENT**</font>

In [2]:
import os, sys

def detect_env():
    try:
        import google.colab
        return 'colab'
    except ImportError:
        return 'local'

ENV = detect_env()

if ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/Taller4_DL_MIAX'
else:
    BASE = os.path.abspath(os.path.join(os.getcwd(), '..'))

SRC = os.path.join(BASE, '01_src_compartido')
PORTFOLIO_DIR = os.path.join(BASE, '11_Portfolio')
for p in [SRC, PORTFOLIO_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f'> Entorno       : {ENV}')
print(f'> BASE          : {BASE}')
print(f'> PORTFOLIO_DIR : {PORTFOLIO_DIR}')

Mounted at /content/drive
> Entorno       : colab
> BASE          : /content/drive/MyDrive/Taller4_DL_MIAX
> PORTFOLIO_DIR : /content/drive/MyDrive/Taller4_DL_MIAX/11_Portfolio


---
## <font color=#0099CC>**2. SELECT MODEL & CONFIGURE**</font>

Change `ACTIVE_MODEL` to switch between available models:
- `'inv_cnn'`   → Conv1D,   V_IN=10, V_OUT=90
- `'inv_mlp'`   → MLP,      V_IN=5,  V_OUT=90
- `'inv_rnn'`   → LSTM,     V_IN=30, V_OUT=90
- `'inv_mixto'` → CNN+RNN+MLP, V_IN=90, V_OUT=90

In [3]:
from config import CFG, MODEL_CATALOG

# ╔══════════════════════════════════════════════════╗
# ║  CHANGE THIS TO SWITCH MODELS                   ║
# ╚══════════════════════════════════════════════════╝
ACTIVE_MODEL = 'inv_cnn'

CFG.apply_model(ACTIVE_MODEL)

# Override TICKERS with the authoritative list from dataset_utils
from dataset_utils import (
    TICKERS, D_FRAC_INV, RANDOM_STATE_VAL,
    load_data, create_dataset, get_partitions_temporal,
)
CFG.TICKERS = list(TICKERS)

print(f'-- Active model: {CFG.ACTIVE_MODEL} --')
print(f'  Architecture : {CFG.ARCH}')
print(f'  V_IN / V_OUT : {CFG.V_IN} / {CFG.V_OUT}')
print(f'  N_ASSETS     : {CFG.N_ASSETS}')
print(f'  REBAL_DAYS   : {CFG.REBAL_DAYS}')
print(f'  Checkpoint   : {CFG.ckpt_path(BASE)}')
print(f'  Exists       : {os.path.isfile(CFG.ckpt_path(BASE))}')

print(f'\n-- Available models --')
for key, info in MODEL_CATALOG.items():
    ckpt_exists = os.path.isfile(
        os.path.join(BASE, '08_results', 'checkpoints',
                     f'{info["modelo"]}_vin{info["v_in"]}_vout{info["v_out"]}_best.weights.h5')
    )
    status = 'READY' if ckpt_exists else 'not found'
    marker = '  <<' if key == ACTIVE_MODEL else ''
    print(f'  {key:15s}  arch={info["arch"]:6s}  vin={info["v_in"]:2d}  [{status}]{marker}')

-- Active model: inv_cnn --
  Architecture : cnn
  V_IN / V_OUT : 10 / 90
  N_ASSETS     : 23
  REBAL_DAYS   : 90
  Checkpoint   : /content/drive/MyDrive/Taller4_DL_MIAX/08_results/checkpoints/inv_cnn_vin10_vout90_best.weights.h5
  Exists       : True

-- Available models --
  inv_cnn          arch=cnn     vin=10  [READY]  <<
  inv_mlp          arch=mlp     vin= 5  [READY]
  inv_rnn          arch=rnn     vin=30  [READY]
  inv_mixto        arch=mixto   vin=90  [not found]


---
## <font color=#0099CC>**3. RECONSTRUCT TRAINING SCALER**</font>

Re-run the exact training data pipeline to recover the scaler.
**Critical**: `V_IN` and `V_OUT` must match what the scaler was trained with.

In [4]:
import numpy as np

data, df = load_data(d_frac=D_FRAC_INV, verbose=True)
print(f'> Datos : {data.shape[0]:,} dias x {data.shape[1]} activos')

X, Y = create_dataset(data, CFG.V_IN, CFG.V_OUT, verbose=True)

X_tr, X_val, X_test, Y_tr, Y_val, Y_test, scaler = get_partitions_temporal(
    X, Y, v_in=CFG.V_IN, v_out=CFG.V_OUT,
    scaler=CFG.SCALER, return_scaler=True,
    random_state=RANDOM_STATE_VAL, verbose=True,
)

print(f'\n> Scaler type  : {type(scaler).__name__}')
print(f'> Scaler mean  : {scaler.mean_[:5].round(6)}  ...')
print(f'> Scaler scale : {scaler.scale_[:5].round(6)}  ...')
print(f'> Shapes: X_tr {X_tr.shape}  X_val {X_val.shape}  X_test {X_test.shape}')

> Descargando datos desde 1945-01-01 hasta hoy...


[*********************100%***********************]  23 of 23 completed


> Frac-diff aplicada — d=0.4, L=1458 pesos, 1457 filas iniciales descartadas (~8.99% del histórico).
> Shape resultante: (14746, 23)
> Datos : 14,746 dias x 23 activos
> Dataset creado — X: (14647, 10, 23), Y: (14647, 23)
> Scaler: standard fitted on X_tr only — applied to train/val/test
> Split temporal con embargo  (v_in=10, v_out=90, embargo=100)
  Train   [     0,  11517)  ->   11517 muestras
  Val     [ 11617,  13082)  ->    1465 muestras  (gap previo: 100)
  Test    [ 13182,  14647)  ->    1465 muestras  (gap previo: 100)
  Descartadas por embargo: 200 muestras  (1.37% del total)

> Scaler type  : StandardScaler
> Scaler mean  : [0.062904 0.078939 0.075525 0.031486 0.069575]  ...
> Scaler scale : [0.054119 0.087925 0.063352 0.061237 0.066451]  ...
> Shapes: X_tr (11517, 10, 23)  X_val (1465, 10, 23)  X_test (1465, 10, 23)


---
## <font color=#0099CC>**4. BUILD MODEL & LOAD WEIGHTS**</font>

In [5]:
import tensorflow as tf
print(f'> TensorFlow : {tf.__version__}')
print(f'> GPU        : {tf.config.list_physical_devices("GPU")}\n')

from model_builder import build_and_load_model

model = build_and_load_model(CFG, BASE)
model.summary()

# Verify MAE on test set
from metrics_utils import calc_mae_all
import metrics_utils
metrics_utils.BASE_DRIVE = BASE

maes = calc_mae_all(model, X_tr, Y_tr, X_val, Y_val, X_test, Y_test)
print(f'\n> MAE  train={maes["train"]:.5f}  val={maes["val"]:.5f}  test={maes["test"]:.5f}')

> TensorFlow : 2.20.0
> GPU        : []

> Model     : inv_cnn (cnn, v_in=10, v_out=90)
> Checkpoint: inv_cnn_vin10_vout90_best.weights.h5
> Parameters: 39,703


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 26 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 10, 64)         │         4,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 10, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 10, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 10, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 10, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 10, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 23)             │         1,495 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 39,703 (155.09 KB)

 Trainable params: 39,319 (153.59 KB)

 Non-trainable params: 384 (1.50 KB)


> MAE  train=0.01865  val=0.02156  test=0.02598


---
## <font color=#0099CC>**5. DOWNLOAD PRICES & PREPARE DATA**</font>

In [6]:
import pandas as pd
from data_loader import download_prices, get_trading_dates, compute_rebalancing_dates, prepare_log_prices

# Frac-diff data from training pipeline (already covers 2025)
fd_full = pd.DataFrame(data, index=df.index[-len(data):], columns=TICKERS)
print(f'> Frac-diff data: {fd_full.shape[0]:,} days, {fd_full.index[0].date()} -> {fd_full.index[-1].date()}')

# Download raw prices for FFD inversion
dl_start = str(pd.Timestamp(CFG.BT_START) - pd.DateOffset(years=CFG.LOOKBACK_YEARS))
dl_end   = str(pd.Timestamp(CFG.BT_END) + pd.DateOffset(days=5))

prices_df = download_prices(CFG.TICKERS, start=dl_start, end=dl_end, verbose=True)
log_prices = prepare_log_prices(prices_df)

# Trading and rebalancing dates
trading_dates = get_trading_dates(prices_df, CFG.BT_START, CFG.BT_END)
rebal_dates = compute_rebalancing_dates(trading_dates, CFG.REBAL_DAYS)

print(f'\n> Backtest trading days : {len(trading_dates)}')
print(f'> Rebalancing dates     : {[d.date() for d in rebal_dates]}')

> Frac-diff data: 14,746 days, 1967-10-12 -> 2026-05-19
> yfinance version : 0.2.66
> Downloading 23 tickers individually (2015-01-01 -> 2026-01-05) ...
  [ 1/23]   AEP  2,767 days  [2015-01-02 -> 2026-01-02]
  [ 2/23]    BA  2,767 days  [2015-01-02 -> 2026-01-02]
  [ 3/23]   CAT  2,767 days  [2015-01-02 -> 2026-01-02]
  [ 4/23]   CNP  2,767 days  [2015-01-02 -> 2026-01-02]
  [ 5/23]   CVX  2,767 days  [2015-01-02 -> 2026-01-02]
  [ 6/23]   DIS  2,767 days  [2015-01-02 -> 2026-01-02]
  [ 7/23]   DTE  2,767 days  [2015-01-02 -> 2026-01-02]
  [ 8/23]    ED  2,767 days  [2015-01-02 -> 2026-01-02]
  [ 9/23]    GD  2,767 days  [2015-01-02 -> 2026-01-02]
  [10/23]    GE  2,767 days  [2015-01-02 -> 2026-01-02]
  [11/23]   HON  2,767 days  [2015-01-02 -> 2026-01-02]
  [12/23]   HPQ  2,767 days  [2015-01-02 -> 2026-01-02]
  [13/23]   IBM  2,767 days  [2015-01-02 -> 2026-01-02]
  [14/23]    IP  2,767 days  [2015-01-02 -> 2026-01-02]
  [15/23]   JNJ  2,767 days  [2015-01-02 -> 2026-01-02]
  [16/2

---
## <font color=#0099CC>**6. RUN PREDICTIONS AT EACH REBALANCING DATE**</font>

In [7]:
from frac_diff_utils import ffd_weights, ffd_invert_expected_return

weights = ffd_weights(CFG.D_FRAC, CFG.FFD_THRESHOLD)
print(f'> FFD weights: L={len(weights)}')

predictions_rows = []
exp_returns_rows = []

for i, rebal_date in enumerate(rebal_dates):
    print(f'\n== Rebalancing [{i}]: {rebal_date.date()} ==')

    # Step 1: V_IN-day window of frac-diff ending at rebal_date
    fd_up_to = fd_full.loc[:rebal_date]
    if len(fd_up_to) < CFG.V_IN:
        print(f'  WARNING: only {len(fd_up_to)} frac-diff days available, need {CFG.V_IN}. Skipping.')
        continue
    window_fd = fd_up_to.iloc[-CFG.V_IN:].values

    # Step 2: Scale with training scaler
    window_scaled = scaler.transform(window_fd)

    # Step 3: Predict
    X_input = window_scaled[np.newaxis, ...]
    pred_avg_fd = model.predict(X_input, verbose=0).squeeze()

    # Store raw predictions
    row_pred = {'rebal_date': rebal_date.strftime('%Y-%m-%d')}
    for j, t in enumerate(TICKERS):
        row_pred[t] = float(pred_avg_fd[j])
    predictions_rows.append(row_pred)

    # Step 4: FFD inversion -> expected returns
    lp_up_to = log_prices.loc[:rebal_date].values
    exp_ret = ffd_invert_expected_return(
        pred_avg_fracdiff=pred_avg_fd,
        log_prices_history=lp_up_to,
        weights=weights,
        horizon=CFG.V_OUT,
    )

    row_ret = {'rebal_date': rebal_date.strftime('%Y-%m-%d')}
    for j, t in enumerate(TICKERS):
        row_ret[t] = float(exp_ret[j])
    exp_returns_rows.append(row_ret)

    # Print top-K
    ranking = np.argsort(exp_ret)[::-1]
    print(f'  Top {CFG.TOP_K} by expected return:')
    for rank in range(min(CFG.TOP_K, CFG.N_ASSETS)):
        idx = ranking[rank]
        print(f'    #{rank+1}  {TICKERS[idx]:>5s}  pred_fd={pred_avg_fd[idx]:+.6f}  exp_ret={exp_ret[idx]:+.4%}')

print(f'\n> Predictions computed for {len(predictions_rows)} rebalancing dates.')

> FFD weights: L=1458

== Rebalancing [0]: 2025-01-02 ==
  Top 5 by expected return:
    #1     KO  pred_fd=+0.169832  exp_ret=+13.5676%
    #2    JNJ  pred_fd=+0.190221  exp_ret=+11.1544%
    #3     BA  pred_fd=+0.189502  exp_ret=+3.7840%
    #4    MRK  pred_fd=+0.167298  exp_ret=+2.6598%
    #5    CVX  pred_fd=+0.186746  exp_ret=+2.5956%

== Rebalancing [1]: 2025-05-14 ==
  Top 5 by expected return:
    #1    MRK  pred_fd=+0.160691  exp_ret=+21.7796%
    #2    JNJ  pred_fd=+0.188093  exp_ret=+8.5599%
    #3    CVX  pred_fd=+0.186027  exp_ret=+4.7904%
    #4     KO  pred_fd=+0.165012  exp_ret=+1.9359%
    #5     PG  pred_fd=+0.185552  exp_ret=+0.6150%

== Rebalancing [2]: 2025-09-23 ==
  Top 5 by expected return:
    #1    MRK  pred_fd=+0.152843  exp_ret=+1.6715%
    #2     KO  pred_fd=+0.157082  exp_ret=-0.0376%
    #3     PG  pred_fd=+0.179806  exp_ret=-0.9367%
    #4    DIS  pred_fd=+0.165272  exp_ret=-5.6518%
    #5    HPQ  pred_fd=+0.115503  exp_ret=-6.2985%

> Predictions comput

---
## <font color=#0099CC>**7. SAVE TO CSV**</font>

Files are saved with the model name in the filename so you can run inference
for multiple models and compare them in the backtest.

In [8]:
out_dir = os.path.join(BASE, '11_Portfolio')
os.makedirs(out_dir, exist_ok=True)

model_tag = CFG.ACTIVE_MODEL  # e.g. 'inv_cnn'

# 1. Raw predictions (average daily frac-diff)
pred_df = pd.DataFrame(predictions_rows).set_index('rebal_date')
pred_path = os.path.join(out_dir, f'predictions_{model_tag}.csv')
pred_df.to_csv(pred_path)
print(f'> Saved: {pred_path}')
print(pred_df.to_string())

# 2. Expected returns (after FFD inversion)
ret_df = pd.DataFrame(exp_returns_rows).set_index('rebal_date')
ret_path = os.path.join(out_dir, f'expected_returns_{model_tag}.csv')
ret_df.to_csv(ret_path)
print(f'\n> Saved: {ret_path}')
print(ret_df.to_string())

# 3. Scaler parameters for verification
scaler_df = pd.DataFrame({
    'ticker': list(TICKERS),
    'scaler_mean': scaler.mean_,
    'scaler_scale': scaler.scale_,
})
scaler_path = os.path.join(out_dir, f'scaler_params_{model_tag}.csv')
scaler_df.to_csv(scaler_path, index=False)
print(f'\n> Saved: {scaler_path}')

# 4. Model metadata (so the backtest knows what it's loading)
import json
meta = {
    'active_model': CFG.ACTIVE_MODEL,
    'arch': CFG.ARCH,
    'v_in': CFG.V_IN,
    'v_out': CFG.V_OUT,
    'd_frac': CFG.D_FRAC,
    'n_assets': CFG.N_ASSETS,
    'tickers': list(TICKERS),
    'rebal_dates': [d.strftime('%Y-%m-%d') for d in rebal_dates],
    'mae_train': maes['train'],
    'mae_val': maes['val'],
    'mae_test': maes['test'],
}
meta_path = os.path.join(out_dir, f'model_meta_{model_tag}.json')
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)
print(f'\n> Saved: {meta_path}')

print(f'\n{"="*50}')
print(f'Done! Files saved for model: {model_tag}')
print(f'  predictions_{model_tag}.csv')
print(f'  expected_returns_{model_tag}.csv')
print(f'  scaler_params_{model_tag}.csv')
print(f'  model_meta_{model_tag}.json')
print(f'\nThe backtest notebook needs: expected_returns_{model_tag}.csv')

> Saved: /content/drive/MyDrive/Taller4_DL_MIAX/11_Portfolio/predictions_inv_cnn.csv
                 AEP        BA       CAT       CNP       CVX       DIS       DTE        ED        GD        GE       HON       HPQ       IBM        IP       JNJ        KO        KR       MMM        MO       MRK       MSI        PG       XOM
rebal_date                                                                                                                                                                                                                                      
2025-01-02  0.150123  0.189502  0.185313  0.125419  0.186746  0.174186  0.160580  0.153264  0.183514  0.212788  0.170112  0.117105  0.189511  0.137840  0.190221  0.169832  0.138947  0.176580  0.148527  0.167298  0.178254  0.187448  0.175228
2025-05-14  0.148977  0.188502  0.188358  0.122806  0.186027  0.170758  0.157780  0.153595  0.186023  0.210350  0.170052  0.116862  0.190496  0.134737  0.188093  0.165012  0.135105  0.176114  

---
## <font color=#0099CC>**8. QUICK VERIFICATION**</font>

Compare expected vs actual returns for each window (if 2025 data is available).

In [9]:
from scipy.stats import spearmanr

for i, rebal_date in enumerate(rebal_dates):
    rebal_idx = list(trading_dates).index(rebal_date)
    end_idx = min(rebal_idx + CFG.V_OUT, len(trading_dates) - 1)
    if end_idx <= rebal_idx:
        continue
    end_date = trading_dates[end_idx]
    actual_days = end_idx - rebal_idx

    p_start = prices_df.loc[rebal_date].values
    p_end   = prices_df.loc[end_date].values
    actual_ret = (p_end / p_start) - 1
    exp_ret = ret_df.iloc[i].values

    corr, pval = spearmanr(exp_ret, actual_ret)

    print(f'Window [{i}]: {rebal_date.date()} -> {end_date.date()} ({actual_days}d)')
    print(f'  Spearman rho = {corr:.4f}  (p={pval:.4f})')

    exp_top_k = set(np.argsort(exp_ret)[-CFG.TOP_K:])
    act_top_k = set(np.argsort(actual_ret)[-CFG.TOP_K:])
    overlap = len(exp_top_k & act_top_k)
    print(f'  Top-{CFG.TOP_K} overlap: {overlap}/{CFG.TOP_K} assets\n')

Window [0]: 2025-01-02 -> 2025-05-14 (90d)
  Spearman rho = 0.1472  (p=0.5026)
  Top-5 overlap: 1/5 assets

Window [1]: 2025-05-14 -> 2025-09-23 (90d)
  Spearman rho = -0.1314  (p=0.5500)
  Top-5 overlap: 1/5 assets

Window [2]: 2025-09-23 -> 2025-12-31 (69d)
  Spearman rho = 0.0109  (p=0.9607)
  Top-5 overlap: 1/5 assets

